In [1]:
import os
import re
import json
import pickle
import tqdm

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
TAGS_LIMIT = 32

In [4]:
SEED = 23
TEMPERATURE = 0.1

In [5]:
MODEL = "google/gemini-2.5-flash"

In [6]:
output_dir_name = MODEL.split("/")[1]

In [7]:
with open(f"{output_dir_name}/tags_set_processed.pkl", "rb") as f:
    tags_set = pickle.load(f)

In [8]:
with open(f"{output_dir_name}/titles_with_tags_dict_processed.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [9]:
len(titles_with_tags)

1147

In [10]:
data = []

In [11]:
for title, tags in titles_with_tags.items():
    for tag in tags:
        data.append((title, tag))

In [12]:
df = pd.DataFrame(data=data, columns=["title", "tag"])

In [13]:
df

,title,tag
0,Исследование приоритетов и механизмов реализац...,BRICS
1,Исследование приоритетов и механизмов реализац...,development_studies
2,Исследование приоритетов и механизмов реализац...,economics
3,Исследование приоритетов и механизмов реализац...,politics
4,Исследование приоритетов и механизмов реализац...,russian_studies
...,...,...
4620,Актив Центра развития карьеры - профориентацио...,psychology
4621,Актив Центра развития карьеры - профориентацио...,sociology
4622,Обзор Методологии в Области Регионоведения. Эт...,data_science
4623,Обзор Методологии в Области Регионоведения. Эт...,economic_geography


In [14]:
dfgr = df.groupby(by=["tag"]).agg({"title": "nunique"}).reset_index()
dfgr.columns = ["tag", "count"]
dfgr = dfgr.sort_values(by=["count"], ascending=False).reset_index(drop=True)
dfgr.head(TAGS_LIMIT)

,tag,count
0,humanities,250
1,education,226
2,digital_media,210
3,russian_studies,199
4,culture,143
5,sociology,118
6,politics,102
7,data_science,98
8,marketing,98
9,economics,97


In [15]:
selected_tags = dfgr.head(TAGS_LIMIT)["tag"].to_list()

In [16]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can create artificial user profiles based on student project titles in russian.
Please, analyze given list of student project titles in russian and create one possible user profile with its own preferences in a provided field of science.
Also select several relevant tags from a tags set, up to 5 most relevant tags.
Return stricly JSON output, consistin of user name, brief user bio and relevant tags list.
For example.
Input data:
```json
{"student_project_titles": ["Исследование методов решения задачи линейной регрессии", "Исследование методов решения задачи линейной классификации", "Исследование методов решения задачи обучения с подкреплением"], "all_possible_tags": ["machine_learning", "computer_science", "economics", "politics"]}
```
Your output:
```json
{"machine_learning_researcher_classic_algorithms": {"bio": "A user, which is interested in machine learning and classic algorithms in particular", "tags": ["machine_learning", "computer_science"]}}
```
"""

In [17]:
messages = []

In [18]:
for tag in selected_tags:
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "student_project_titles": list(
                            df[df["tag"] == tag]["title"].unique()
                        ),
                        "all_possible_tags": list(tags_set),
                    },
                    ensure_ascii=False,
                ),
            },
        )
    )

In [19]:
client = OpenAI(
    api_key=os.environ.get("KODIK_API_KEY"),
    base_url="https://api.kodikrouter.ru/v1",
)

In [20]:
answers = {}

In [21]:
for tag, message in tqdm.tqdm(list(zip(selected_tags, messages))):

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=message,
            seed=SEED,
            temperature=TEMPERATURE,
        )

        content = response.choices[0].message.content

        json_match = re.search(r"```json\s*(.*?)\s*```", content, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_str = content

        answers.update(json.loads(json_str))

    except Exception as e:

        print(f"something went wrong with tag {tag} - {e}")

100%|██████████| 32/32 [00:39<00:00,  1.24s/it]


In [22]:
len(answers)

31

In [23]:
answers[list(answers.keys())[0]]

{'bio': 'A user interested in various aspects of culture, history, and social phenomena, with a particular focus on non-Western cultures, literature, and media.',
 'tags': ['arts_and_culture',
  'middle_eastern_studies',
  'asian_studies',
  'folklore',
  'postcolonialism']}

In [24]:
answers[list(answers.keys())[-1]]

{'bio': 'A user interested in the full lifecycle of mobile application development, from initial concept and design to implementation and monetization strategies. They focus on creating user-centric applications across various platforms (Android, iOS, cross-platform) and often incorporate features like social networking, financial management, educational tools, and AI-powered functionalities.',
 'tags': ['mobile_development',
  'android',
  'react_native',
  'flutter',
  'kotlin']}

In [25]:
output = {}

for k in answers.keys():
    output[k.lower()] = answers[k]

In [26]:
sorted(output.keys())

['african_and_middle_eastern_studies_researcher',
 'ai_in_education_and_law_enthusiast',
 'ai_in_education_innovator',
 'art_and_culture_enthusiast',
 'business_strategist',
 'cultural_cinema_enthusiast',
 'cultural_exchange_coordinator',
 'cultural_studies_enthusiast',
 'data_management_specialist',
 'east_asia_cultural_specialist',
 'educational_content_developer',
 'event_organizer_and_community_builder',
 'financial_analyst_macro_and_micro',
 'geopolitical_analyst',
 'geopolitical_analyst_brics_middle_east',
 'geopolitical_analyst_global_economy',
 'global_economic_analyst',
 'legal_innovation_and_education_enthusiast',
 'linguistic_ai_developer',
 'linguistics_and_cultural_studies_enthusiast',
 'marketing_and_media_strategist',
 'media_and_communications_specialist',
 'media_content_creator',
 'mobile_app_developer',
 'project_manager_and_event_organizer',
 'psychology_researcher_mental_health_and_wellbeing',
 'sociology_and_cultural_studies_enthusiast',
 'software_developer_for_e

In [27]:
with open(f"{output_dir_name}/artificial_profiles.json", "w") as f:
    json.dump(output, f)

In [31]:
max([len(x["tags"]) for x in output.values()])

5

In [32]:
min([len(x["tags"]) for x in output.values()])

5